# 1MTR58 – LAB2 – Experiencia 2
## Procesamiento y caracterización de señales EOG

Este notebook acompaña la **sesión de laboratorio**. El objetivo es llegar hasta un
`DataFrame` de características listo para el posterior entrenamiento de modelos.

Flujo:

**Carga → EDA → EOGh/EOGv → filtrado → etiquetas → segmentación → ventaneo → features**

> El entrenamiento de modelos **no forma parte de este notebook**.


## 0. Configuración

El dataset debe estar dentro de la carpeta `data/` del repositorio con nombres como:

- `S1-EOG.csv`
- `S1-ControlSignal.csv`
- ...
- `S7-EOG.csv`
- `S7-ControlSignal.csv`

Durante la sesión se trabajará primero con un sujeto. Cambie `SUBJECT_ID` si el JP lo indica.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt

DATA_DIR = Path("data")
SUBJECT_ID = "S1"
FS = 256

EOG_PATH = DATA_DIR / f"{SUBJECT_ID}-EOG.csv"
CS_PATH = DATA_DIR / f"{SUBJECT_ID}-ControlSignal.csv"

print("EOG:", EOG_PATH)
print("ControlSignal:", CS_PATH)


## 1. Carga y análisis exploratorio

Revise dimensiones, tipos de variables, valores faltantes, estadísticas descriptivas,
señales V1–V4 y matriz de correlación.


In [ ]:
df_eog = pd.read_csv(EOG_PATH)
df_cs = pd.read_csv(CS_PATH)

# Eliminar columnas de índice exportadas por error
df_eog = df_eog.loc[:, ~df_eog.columns.str.contains(r"^Unnamed")].copy()
df_cs = df_cs.loc[:, ~df_cs.columns.str.contains(r"^Unnamed")].copy()

df_eog.columns = [c.strip() for c in df_eog.columns]
df_cs.columns = [c.strip() for c in df_cs.columns]

if "ControlSignal" not in df_cs.columns:
    if len(df_cs.columns) == 1:
        df_cs.columns = ["ControlSignal"]
    else:
        raise ValueError("No se encontró la columna ControlSignal.")

required = {"V1", "V2", "V3", "V4"}
missing = required - set(df_eog.columns)
if missing:
    raise ValueError(f"Faltan columnas EOG: {sorted(missing)}")

print("Dimensión EOG:", df_eog.shape)
print("Dimensión ControlSignal:", df_cs.shape)
print("\nTipos:")
print(df_eog.dtypes)
print("\nNaN:")
print(df_eog.isna().sum())
print("\nEstadísticas:")
display(df_eog[["V1","V2","V3","V4"]].describe())


In [ ]:
# Visualización de V1–V4
time_eog = np.arange(len(df_eog)) / FS

fig, ax = plt.subplots(figsize=(13,5))
for col in ["V1","V2","V3","V4"]:
    ax.plot(time_eog, df_eog[col], label=col, alpha=0.75)
ax.set_title(f"Señales EOG crudas – {SUBJECT_ID}")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# Matriz de correlación sin seaborn
corr = df_eog[["V1","V2","V3","V4"]].corr()
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(corr.values, vmin=-1, vmax=1)
ax.set_xticks(range(4), corr.columns)
ax.set_yticks(range(4), corr.index)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="Correlación")
ax.set_title("Matriz de correlación")
plt.show()


## 2. Integración con etiquetas y construcción de EOGh/EOGv

Cada fila de `ControlSignal` etiqueta una muestra del registro EOG. Ambos archivos se
recortan al menor tamaño antes de unirlos.


In [ ]:
n = min(len(df_eog), len(df_cs))
df = pd.concat(
    [
        df_eog.iloc[:n].reset_index(drop=True),
        df_cs[["ControlSignal"]].iloc[:n].reset_index(drop=True),
    ],
    axis=1,
)

df["subject_id"] = SUBJECT_ID
df["Tiempo"] = np.arange(len(df)) / FS

df["EOGh"] = df["V3"] - df["V4"]
df["EOGv"] = df["V1"] - df["V2"]

print("DataFrame integrado:", df.shape)
print("\nClases:")
print(df["ControlSignal"].value_counts().sort_index())
display(df.head())


## 3. Preprocesamiento

Se aplica un filtro pasabanda de **0.1–30 Hz** para reducir la deriva de línea base
y componentes de alta frecuencia. Compare señal cruda y filtrada.


In [ ]:
def bandpass_filter(x, fs=256, lowcut=0.1, highcut=30.0, order=4):
    x = np.asarray(x, dtype=float)
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
    return filtfilt(b, a, x)

df["EOGh_filt"] = bandpass_filter(df["EOGh"], fs=FS)
df["EOGv_filt"] = bandpass_filter(df["EOGv"], fs=FS)

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(df["Tiempo"], df["EOGh"], label="EOGh cruda", alpha=0.35)
ax.plot(df["Tiempo"], df["EOGh_filt"], label="EOGh filtrada", linewidth=1)
ax.set_title(f"EOGh: cruda vs filtrada – {SUBJECT_ID}")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(df["Tiempo"], df["EOGv"], label="EOGv cruda", alpha=0.35)
ax.plot(df["Tiempo"], df["EOGv_filt"], label="EOGv filtrada", linewidth=1)
ax.set_title(f"EOGv: cruda vs filtrada – {SUBJECT_ID}")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 4. Segmentación por clase

Primero se detectan bloques continuos de una misma clase. Esto evita generar ventanas
que mezclen estados distintos.


In [ ]:
df["block_id"] = (df["ControlSignal"] != df["ControlSignal"].shift()).cumsum()

segments = (
    df.groupby("block_id")
      .agg(
          label=("ControlSignal", "first"),
          start_idx=("ControlSignal", lambda x: x.index[0]),
          end_idx=("ControlSignal", lambda x: x.index[-1]),
          n_samples=("ControlSignal", "size"),
          start_time=("Tiempo", "first"),
          end_time=("Tiempo", "last"),
      )
      .reset_index(drop=True)
)

segments["duration_s"] = segments["n_samples"] / FS

print("Número total de segmentos:", len(segments))
print("\nSegmentos por clase:")
print(segments["label"].value_counts().sort_index())
display(segments.head(10))


## 5. Ventaneo

Configuración recomendada:

- ventana = 1 s = 256 muestras
- overlap = 50 %
- paso = 128 muestras


In [ ]:
WINDOW_SEC = 1.0
OVERLAP_SEC = 0.5

WIN_SIZE = int(WINDOW_SEC * FS)
STEP = int((WINDOW_SEC - OVERLAP_SEC) * FS)

records = []

for _, seg in segments.iterrows():
    label = int(seg["label"])
    start_seg = int(seg["start_idx"])
    end_seg = int(seg["end_idx"]) + 1

    if end_seg - start_seg < WIN_SIZE:
        continue

    for start in range(start_seg, end_seg - WIN_SIZE + 1, STEP):
        end = start + WIN_SIZE

        # Verificación: una ventana debe contener una sola clase
        if df.loc[start:end-1, "ControlSignal"].nunique() != 1:
            continue

        records.append({
            "subject_id": SUBJECT_ID,
            "label": label,
            "start_idx": start,
            "end_idx": end - 1,
            "EOGh_window": df.loc[start:end-1, "EOGh_filt"].to_numpy(),
            "EOGv_window": df.loc[start:end-1, "EOGv_filt"].to_numpy(),
        })

windows_df = pd.DataFrame(records)

print("Total de ventanas:", len(windows_df))
print("\nVentanas por clase:")
print(windows_df["label"].value_counts().sort_index())
display(windows_df[["subject_id","label","start_idx","end_idx"]].head())


## 6. Extracción y visualización de características

Se extraen las mismas características para EOGh y EOGv.


In [ ]:
def waveform_length(x):
    x = np.asarray(x)
    return np.sum(np.abs(np.diff(x)))

def extract_features(x, prefix):
    x = np.asarray(x, dtype=float)
    return {
        f"{prefix}_MAV": np.mean(np.abs(x)),
        f"{prefix}_RMS": np.sqrt(np.mean(x**2)),
        f"{prefix}_STD": np.std(x),
        f"{prefix}_WL": waveform_length(x),
        f"{prefix}_MEAN": np.mean(x),
        f"{prefix}_MAX": np.max(x),
        f"{prefix}_MIN": np.min(x),
        f"{prefix}_PTP": np.ptp(x),
    }

feature_rows = []

for _, row in windows_df.iterrows():
    feats = {}
    feats.update(extract_features(row["EOGh_window"], "EOGh"))
    feats.update(extract_features(row["EOGv_window"], "EOGv"))
    feats["subject_id"] = row["subject_id"]
    feats["label"] = row["label"]
    feats["start_idx"] = row["start_idx"]
    feats["end_idx"] = row["end_idx"]
    feature_rows.append(feats)

features_df = pd.DataFrame(feature_rows)

print("Tamaño del dataframe final:", features_df.shape)
print("\nVentanas por clase:")
print(features_df["label"].value_counts().sort_index())
display(features_df.head())


In [ ]:
# Comparación de algunas características por clase
feature_plot_cols = ["EOGh_MAV", "EOGh_RMS", "EOGh_STD", "EOGv_MAV", "EOGv_RMS", "EOGv_STD"]
summary = features_df.groupby("label")[feature_plot_cols].mean()

display(summary)

summary.plot(kind="bar", figsize=(11,5))
plt.title("Características promedio por clase")
plt.xlabel("Clase")
plt.ylabel("Valor promedio")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Checkpoint de la sesión

Antes de finalizar, el grupo debe mostrar al JP:

1. Señales EOG originales y procesadas.
2. Segmentos etiquetados.
3. Cantidad de ventanas por clase.
4. `features_df` final.
5. Una comparación gráfica de características entre clases.

**Aquí termina el notebook de apoyo para la sesión.**
El entrenamiento de modelos corresponde al Informe.
